In [6]:
# =========================================================================
#  DEPENDENCIAS — EJECUTAR ESTA CELDA PRIMERO
# -------------------------------------------------------------------------
#  Revisa paquete por paquete e instala UNICAMENTE los que falten.
#  En Google Colab casi todos vienen preinstalados, asi que lo normal es
#  que esta celda no descargue nada y termine en un segundo.
# =========================================================================
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

# modulo que se importa  ->  nombre con el que se instala desde pip
REQUISITOS = {
    "numpy": "numpy",                     # arreglos numericos
    "pandas": "pandas",                   # dataframes tabulares
    "pyarrow": "pyarrow",                 # lectura/escritura de .parquet
    "sklearn": "scikit-learn",            # splits, scaler, PCA, t-SNE, K-Means
    "matplotlib": "matplotlib",           # graficos del EDA
    "psutil": "psutil",                   # medicion de RAM
    "joblib": "joblib",                   # guardado del StandardScaler
    "torch": "torch",                     # tensores y redes neuronales
    "ydata_profiling": "ydata-profiling",  # auditoria de particiones
}

faltantes = [pip_name for modulo, pip_name in REQUISITOS.items()
             if importlib.util.find_spec(modulo) is None]

if faltantes:
    print(f"Faltan {len(faltantes)} paquete(s): {', '.join(faltantes)}")
    print("Instalando (puede tardar un par de minutos)...\n")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)
    print("\nInstalacion terminada.")
    print("Si Colab pide REINICIAR el entorno de ejecucion, reinicia y vuelve a")
    print("ejecutar desde aqui: la descarga del dataset se reanuda sola.")
else:
    print("Todas las dependencias ya estaban instaladas: no se descargo nada.")

print("\nVersiones en uso")
print("-" * 36)
for pip_name in REQUISITOS.values():
    try:
        print(f"  {pip_name:<18} {metadata.version(pip_name)}")
    except metadata.PackageNotFoundError:
        print(f"  {pip_name:<18} (no detectada)")

Todas las dependencias ya estaban instaladas: no se descargo nada.

Versiones en uso
------------------------------------
  numpy              2.3.5
  pandas             2.3.3
  pyarrow            26.0.1
  scikit-learn       1.9.1
  matplotlib         3.10.0
  psutil             5.9.8
  joblib             1.6.0
  torch              2.15.0
  ydata-profiling    4.18.4


# Laboratorio 1 — Fase A: Carga Masiva, Particiones y EDA
### Opción B: Predicción Científica — *HIGGS Boson Dataset* (UCI ML Repository)

**Curso:** Fundamentos de Aprendizaje Profundo y Aplicaciones — 2-2026
**Grupo:** _(completar)_ · **Integrantes:** _(completar)_

---

## 1. Justificación del dataset y del problema científico

El **HIGGS Boson Dataset** (Baldi, Sadowski & Whiteson, *Nature Communications* 5, 2014) contiene
**11.000.000 de colisiones simuladas** de partículas en un acelerador. La tarea es **clasificación
binaria**: distinguir un evento de **señal** (producción de un bosón de Higgs) de uno de **fondo**
(*background*), es decir, un proceso físico distinto que produce exactamente los mismos productos
de decaimiento observables.

Es un problema genuinamente difícil: las variables son continuas, están fuertemente correlacionadas
entre sí y las diferencias entre ambas clases son sutiles — justamente el escenario donde el paper
original mostró que una red profunda supera a los métodos tradicionales de la física de altas
energías.

### Descripción de las 28 variables, una por una

La primera columna del archivo es la **etiqueta** y las 28 siguientes son las variables
predictoras.

| # | Columna | Qué es |
|---|---|---|
| 0 | `label` | **Lo que hay que predecir.** `1` = señal (la colisión produjo un bosón de Higgs), `0` = fondo (otro proceso físico que deja los mismos productos observables) |

#### Cinemáticas de bajo nivel (21) — lo que mide directamente el detector

Describen la trayectoria y la energía de cada partícula detectada. Cada dirección necesita
tres números: cuánta energía lleva (`pt`), cuán inclinada va respecto al haz (`eta`) y hacia
qué lado del círculo apunta (`phi`).

| # | Columna | Qué es |
|---|---|---|
| 1 | `lepton_pT` | Momento transversal del leptón (electrón o muón): su momento proyectado en el plano perpendicular al haz. Es la magnitud que mejor mide "cuánta energía" lleva la partícula |
| 2 | `lepton_eta` | Pseudorapidez del leptón: qué tan inclinada va su trayectoria respecto al eje del haz. η = 0 es perpendicular al haz; valores grandes significan que salió casi pegado al haz |
| 3 | `lepton_phi` | Ángulo azimutal del leptón: hacia qué lado apunta dentro del plano transversal, en radianes de −π a π |
| 4 | `missing_energy_magnitude` | Energía transversal faltante. Como el momento total debe conservarse, un desbalance delata partículas que atravesaron el detector sin interactuar: neutrinos |
| 5 | `missing_energy_phi` | Dirección azimutal de esa energía faltante |
| 6 | `jet1_pt` | Momento transversal del jet 1. Un *jet* es la lluvia de partículas que deja un quark o gluón; vienen ordenados de mayor a menor momento |
| 7 | `jet1_eta` | Pseudorapidez del jet 1 |
| 8 | `jet1_phi` | Ángulo azimutal del jet 1 |
| 9 | `jet1_b_tag` | Etiquetado *b* del jet 1: marca si el jet probablemente nació de un quark *bottom*. Es la pista clave, porque el Higgs decae a un par de quarks *b* |
| 10 | `jet2_pt` | Momento transversal del jet 2 |
| 11 | `jet2_eta` | Pseudorapidez del jet 2 |
| 12 | `jet2_phi` | Ángulo azimutal del jet 2 |
| 13 | `jet2_b_tag` | Etiquetado *b* del jet 2 |
| 14 | `jet3_pt` | Momento transversal del jet 3 |
| 15 | `jet3_eta` | Pseudorapidez del jet 3 |
| 16 | `jet3_phi` | Ángulo azimutal del jet 3 |
| 17 | `jet3_b_tag` | Etiquetado *b* del jet 3 |
| 18 | `jet4_pt` | Momento transversal del jet 4 |
| 19 | `jet4_eta` | Pseudorapidez del jet 4 |
| 20 | `jet4_phi` | Ángulo azimutal del jet 4 |
| 21 | `jet4_b_tag` | Etiquetado *b* del jet 4 |

#### Derivadas de alto nivel (7) — calculadas por los físicos

No aportan información nueva: son **funciones de las 21 anteriores**. Son *masas invariantes*,
es decir, la masa que tendría que haber tenido la partícula original para producir ese conjunto
de productos. Si en un evento los productos "encajan" con la masa de un bosón W, de un quark
top o del Higgs, eso se ve aquí y no en las variables crudas.

| # | Columna | Qué es |
|---|---|---|
| 22 | `m_jj` | Masa invariante de dos jets ligeros: reconstruye el bosón **W** que decayó a hadrones |
| 23 | `m_jjj` | Masa invariante de tres jets: reconstruye el quark **top** hadrónico |
| 24 | `m_lv` | Masa invariante del leptón más el neutrino (la energía faltante): el bosón **W** que decayó a leptones |
| 25 | `m_jlv` | Masa invariante de jet + leptón + neutrino: el quark **top** leptónico |
| 26 | `m_bb` | Masa invariante de los dos jets *b*. **La más discriminante de todas**: en los eventos de señal reconstruye el **bosón de Higgs**, que decae a un par b-antib |
| 27 | `m_wbb` | Masa invariante del sistema W + los dos jets *b* |
| 28 | `m_wwbb` | Masa invariante del sistema completo: los dos W más los dos jets *b* |

> **Sobre las unidades:** en el archivo original los valores ya vienen reescalados, así que los
> momentos y las masas no se leen directamente en GeV. Los rangos reales se ven en el
> `describe()` de la celda 5.

La etiqueta (`label`) es la primera columna: **1 = señal**, **0 = fondo**.

## 2. Cumplimiento de los requisitos del enunciado

| Requisito | Estado |
|---|---|
| Volumen: 200.000 – 1.000.000 registros | ✅ Se extrae una **submuestra aleatoria de 1.000.000 de filas** de los 11M originales |
| Dimensionalidad: ≥ 40 columnas | ⚠️ HIGGS tiene **28 features**. El profesor autorizó expresamente usarlo tal cual, dado que el volumen de observaciones (11M disponibles) compensa la menor dimensionalidad. Además, el enunciado lo lista como ejemplo explícito de la Opción B |
| EDA y no supervisado sobre submuestra del 10% | ✅ Fase A se ejecuta sobre 100.000 filas |
| Auditoría de particiones con `ydata-profiling` | ✅ `compare([train, val, test])` con `minimal=True` |
| Gestión activa de memoria | ✅ Lectura por *chunks*, `float32`, `.parquet`, `del` + `gc.collect()` |
| Checkpoints en Google Drive | ✅ Si hay Drive disponible (Colab o la app de escritorio) los artefactos se escriben ahí; si no, quedan en el repositorio |

## 3. Qué produce este notebook

1. `higgs_1M.parquet` — submuestra de 1M filas.
2. `splits_escalados.npz` — Train/Val/Test ya escalados **sin data leakage**.
3. `scaler.joblib` — el `StandardScaler` ajustado **solo con Train**.
4. `comparacion_particiones.html` — evidencia de ydata-profiling exigida en los entregables.

---
# Celda 1 — Imports, reproducibilidad y utilidades de memoria

In [7]:
import gc
import os
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import torch

warnings.filterwarnings("ignore")

SEED = 42


def set_seed(seed: int = SEED) -> None:
    '''Fija todas las fuentes de aleatoriedad para que el experimento sea reproducible.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def mem_report(tag: str = "") -> None:
    '''Reporta el consumo de RAM del proceso y del sistema.

    La gestion de memoria es un criterio explicito de la rubrica: en Colab disponemos de ~12 GB
    y el dataset original pesa 7.5 GB sin comprimir, por lo que NUNCA se carga completo.
    '''
    rss = psutil.Process(os.getpid()).memory_info().rss / 1e9
    vm = psutil.virtual_memory()
    print(f"[RAM] {tag:<34} proceso: {rss:5.2f} GB | sistema: {vm.used/1e9:5.2f}/{vm.total/1e9:.2f} GB ({vm.percent:.0f}%)")


mem_report("inicio")

Dispositivo: cpu
[RAM] inicio                             proceso:  0.32 GB | sistema: 18.90/34.16 GB (55%)


---
# Celda 2 — Entorno de ejecución y rutas del proyecto

El notebook se adapta solo a tres escenarios, sin tocar una línea:

| Dónde se ejecuta | Dónde quedan los artefactos |
|---|---|
| **Google Colab** | Google Drive, montado en `/content/drive` |
| **Local con Google Drive para escritorio** | La unidad sincronizada (`G:\Mi unidad\Lab1-DL`) |
| **Local sin Drive** | Dentro del repositorio, en carpetas que el `.gitignore` excluye |

En los dos primeros casos los checkpoints terminan en Drive, que es justamente lo que pide el
enunciado: que un corte a mitad del entrenamiento no obligue a empezar de cero.

El `.csv.gz` de 2.6 GB es la excepción y **siempre** se guarda en disco local: es un archivo
temporal que se lee una sola vez, y sincronizarlo a Drive sería gastar cuota para nada.


In [8]:
# -------------------------------------------------------------------------
#  Deteccion del entorno y ubicacion de los artefactos
# -------------------------------------------------------------------------
# Nota: no sirve importlib.util.find_spec("google.colab"), porque LANZA
# ModuleNotFoundError cuando el paquete padre `google` no existe (solo
# devuelve None si existe `google` pero le falta `colab`). Se comprueba
# intentando la importacion, que es barata cuando falla.
try:
    import google.colab  # noqa: F401

    EN_COLAB = True
except ModuleNotFoundError:
    EN_COLAB = False


def raiz_del_repo():
    """Carpeta Lab1-DL, se ejecute el notebook desde notebooks/ o desde la raiz."""
    actual = Path.cwd().resolve()
    for candidata in (actual, *actual.parents):
        if (candidata / "src").is_dir() and (candidata / "notebooks").is_dir():
            return candidata
    return actual


def buscar_drive_local():
    """Ubica 'Mi unidad' de Google Drive para escritorio, si esta montado."""
    for letra in "GHIJKLDEF":
        for nombre in ("Mi unidad", "My Drive"):
            ruta = Path(f"{letra}:/{nombre}")
            try:
                if ruta.is_dir():
                    return ruta
            except OSError:
                continue
    return None


REPO = raiz_del_repo()

if EN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Lab1-DL")
    RAW_GZ = Path("/content/HIGGS.csv.gz")       # disco efimero de Colab
    ENTORNO = "Google Colab, con Drive montado"
else:
    unidad = buscar_drive_local()
    if unidad is not None:
        PROJECT_DIR = unidad / "Lab1-DL"
        ENTORNO = f"Local, con Google Drive para escritorio en {unidad}"
    else:
        PROJECT_DIR = REPO
        ENTORNO = "Local, sin Drive: los artefactos quedan en el repositorio"
    # El .gz de 2.6 GB nunca va a Drive: se lee una vez y es descartable.
    RAW_GZ = REPO / "data" / "raw" / "HIGGS.csv.gz"

PROC_DIR = PROJECT_DIR / "data" / "processed"    # parquet y splits escalados
CKPT_DIR = PROJECT_DIR / "checkpoints"           # pesos .pt (notebooks 02 y 03)
REPORT_DIR = PROJECT_DIR / "reports"             # html de profiling y figuras

for carpeta in (PROC_DIR, CKPT_DIR, REPORT_DIR, RAW_GZ.parent):
    carpeta.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = PROC_DIR / "higgs_1M.parquet"
SPLITS_PATH = PROC_DIR / "splits_escalados.npz"
SCALER_PATH = PROC_DIR / "scaler.joblib"

print(f"Entorno detectado: {ENTORNO}\n")
print(f"  artefactos   -> {PROJECT_DIR}")
print(f"  procesados   -> {PROC_DIR}")
print(f"  checkpoints  -> {CKPT_DIR}")
print(f"  reportes     -> {REPORT_DIR}")
print(f"  dataset .gz  -> {RAW_GZ}")

if not EN_COLAB and buscar_drive_local() is None:
    print("\nAVISO: no se detecto Google Drive para escritorio.")
    print("Para que los checkpoints queden en Drive, instala la app desde")
    print("https://www.google.com/drive/download/ , inicia sesion y re-ejecuta")
    print("esta celda: la unidad sincronizada se detecta sola.")


ModuleNotFoundError: No module named 'google'

---
# Celda 3 — Descarga del dataset (2.6 GB comprimidos)

Escrita en Python puro en vez de `wget`, porque Windows no lo trae instalado.

La función **intenta** reanudar una descarga cortada pidiendo el tramo que falta con la cabecera
HTTP `Range`. Comprobado contra el servidor de UCI: **no soporta reanudación** — responde `200`
en lugar de `206` —, así que si se corta, vuelve a empezar. No es grave: medido en esta máquina,
la descarga va a unos 17 MB/s y los 2.6 GB tardan cerca de 3 minutos.

Si el archivo ya está completo en disco, la celda no descarga nada.


In [ ]:
import urllib.request

URL_HIGGS = "https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz"
TAMANO_MINIMO = 2_500_000_000   # umbral para dar por completa la descarga (~2.6 GB)


def descargar_con_reanudacion(url, destino, intentos=5, bloque=1 << 20):
    """Descarga con reintentos, sin depender de wget.

    Si ya hay bytes en disco pide al servidor solo el tramo que falta (cabecera
    `Range`). Cuando el servidor no soporta reanudacion responde 200 en vez de
    206 y la descarga se reinicia desde cero: es el caso del servidor de UCI,
    verificado. Los reintentos sirven igual ante cortes de red transitorios.
    """
    destino = Path(destino)
    for intento in range(1, intentos + 1):
        descargado = destino.stat().st_size if destino.exists() else 0
        peticion = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
        if descargado:
            peticion.add_header("Range", f"bytes={descargado}-")
            print(f"Reanudando desde {descargado/1e9:.2f} GB (intento {intento}/{intentos})")

        try:
            with urllib.request.urlopen(peticion, timeout=120) as respuesta:
                if descargado and respuesta.status != 206:
                    print("  el servidor no acepto reanudar: se reinicia desde cero")
                    descargado = 0
                modo = "ab" if descargado else "wb"

                inicio = time.time()
                bytes_sesion = 0
                siguiente_aviso = 250 * (1 << 20)

                with open(destino, modo) as salida:
                    while True:
                        trozo = respuesta.read(bloque)
                        if not trozo:
                            break
                        salida.write(trozo)
                        bytes_sesion += len(trozo)
                        if bytes_sesion >= siguiente_aviso:
                            siguiente_aviso += 250 * (1 << 20)
                            total = (descargado + bytes_sesion) / 1e9
                            vel = bytes_sesion / max(time.time() - inicio, 1e-9) / 1e6
                            print(f"  {total:5.2f} GB descargados  ({vel:5.1f} MB/s)")
            return destino

        except Exception as error:
            print(f"  intento {intento} interrumpido: {type(error).__name__}: {error}")

    raise RuntimeError(f"No se pudo completar la descarga tras {intentos} intentos")


if RAW_GZ.exists() and RAW_GZ.stat().st_size >= TAMANO_MINIMO:
    print(f"Ya descargado: {RAW_GZ} ({RAW_GZ.stat().st_size/1e9:.2f} GB)")
else:
    t0 = time.time()
    descargar_con_reanudacion(URL_HIGGS, RAW_GZ)
    print(f"\nDescarga completada en {(time.time()-t0)/60:.1f} min")

print(f"Tamano final: {RAW_GZ.stat().st_size/1e9:.2f} GB")


---
# Celda 4 — Carga por *chunks* y submuestreo a 1.000.000 de filas

**Esta es la celda crítica de memoria del laboratorio.** El archivo descomprimido ocupa ~7.5 GB:
leerlo de una sola vez con `pd.read_csv()` colapsa la sesión de Colab (OOM).

Estrategia:

1. Leer en bloques de 1.000.000 de filas (`chunksize`), de modo que en RAM nunca haya más de un
   bloque a la vez.
2. Forzar `float32` en lugar del `float64` por defecto de pandas → **la mitad de memoria**.
3. Tomar una muestra aleatoria del 9.09 % de **cada** bloque. Al muestrear uniformemente en todos
   los bloques, la submuestra final de 1M de filas es representativa del dataset completo y no
   queda sesgada hacia el inicio del archivo (el orden de las filas en HIGGS no es aleatorio: el
   benchmark original reserva las últimas 500.000 como test).
4. Liberar cada bloque con `del` + `gc.collect()` apenas se usa.

In [ ]:
COLUMNAS = [
    "label",
    "lepton_pT", "lepton_eta", "lepton_phi",
    "missing_energy_magnitude", "missing_energy_phi",
    "jet1_pt", "jet1_eta", "jet1_phi", "jet1_b_tag",
    "jet2_pt", "jet2_eta", "jet2_phi", "jet2_b_tag",
    "jet3_pt", "jet3_eta", "jet3_phi", "jet3_b_tag",
    "jet4_pt", "jet4_eta", "jet4_phi", "jet4_b_tag",
    "m_jj", "m_jjj", "m_lv", "m_jlv", "m_bb", "m_wbb", "m_wwbb",
]
FEATURES = COLUMNAS[1:]           # 28 variables predictoras
N_TOTAL = 11_000_000              # filas del dataset original
N_OBJETIVO = 1_000_000            # tope superior permitido por el enunciado
FRAC = N_OBJETIVO / N_TOTAL       # ~0.0909

if PARQUET_PATH.exists():
    print(f"Submuestra ya existe, se recarga desde disco: {PARQUET_PATH}")
    df = pd.read_parquet(PARQUET_PATH)
else:
    t0 = time.time()
    bloques = []
    lector = pd.read_csv(
        RAW_GZ,
        header=None,
        names=COLUMNAS,
        dtype=np.float32,
        chunksize=1_000_000,
        compression="gzip",
    )

    for i, bloque in enumerate(lector):
        bloques.append(bloque.sample(frac=FRAC, random_state=SEED + i))
        filas = sum(len(b) for b in bloques)
        print(f"  bloque {i+1:2d}/11 procesado | acumulado: {filas:,} filas | {time.time()-t0:5.0f}s")
        del bloque
        gc.collect()

    df = pd.concat(bloques, ignore_index=True)
    del bloques, lector
    gc.collect()

    # Mezcla final y tipos minimos
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    df["label"] = df["label"].astype(np.int8)

    print(f"\nLectura y submuestreo completados en {(time.time()-t0)/60:.1f} min")

print(f"\nDimensiones de la submuestra: {df.shape[0]:,} filas x {df.shape[1]} columnas "
      f"({len(FEATURES)} features + label)")
print(f"Memoria del DataFrame: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
mem_report("tras cargar submuestra 1M")

---
# Celda 5 — Persistencia en formato comprimido (`.parquet`)

El enunciado exige preferir formatos comprimidos sobre `.csv`. Parquet es columnar, conserva los
tipos (`float32` sigue siendo `float32` al releer, cosa que el CSV no garantiza) y ocupa una
fracción del espacio. Guardarlo en Drive significa que las siguientes ejecuciones — y los
notebooks 02 y 03 — parten desde aquí sin volver a descargar los 2.6 GB.

In [ ]:
if not PARQUET_PATH.exists():
    t0 = time.time()
    df.to_parquet(PARQUET_PATH, index=False, compression="snappy")
    print(f"Guardado en {time.time()-t0:.0f}s")

print(f"Parquet: {PARQUET_PATH}")
print(f"Tamano en disco: {PARQUET_PATH.stat().st_size/1e6:.1f} MB "
      f"(vs ~{df.memory_usage(deep=True).sum()/1e6:.0f} MB en RAM)")

# Vista rapida de los datos
display(df.head())
display(df[FEATURES].describe().T[["mean", "std", "min", "max"]].round(3))

---
# Celda 6 — Particiones Train / Validation / Test **sin Data Leakage**

Dos decisiones que la rúbrica evalúa explícitamente:

1. **Estratificación:** `stratify=y` garantiza que la proporción señal/fondo sea idéntica en las
   tres particiones.
2. **El `StandardScaler` se ajusta SOLO con el conjunto de entrenamiento.** Si se ajustara con el
   dataset completo antes de partir, la media y la desviación estándar del Train incorporarían
   información del Test — eso es *data leakage* y vuelve optimistas todas las métricas.
   Validación y Test se **transforman** con los parámetros del Train, nunca se re-ajustan.

Proporción: **70 % Train / 15 % Validation / 15 % Test**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

if SPLITS_PATH.exists():
    print(f"Splits ya existen, se recargan desde disco: {SPLITS_PATH}")
    d = np.load(SPLITS_PATH)
    X_train, y_train = d["X_train"], d["y_train"]
    X_val, y_val = d["X_val"], d["y_val"]
    X_test, y_test = d["X_test"], d["y_test"]
    scaler = joblib.load(SCALER_PATH)
    del d
    gc.collect()
else:
    X = df[FEATURES].to_numpy(dtype=np.float32)
    y = df["label"].to_numpy()

    # 85% temporal / 15% test
    X_tmp, X_test, y_tmp, y_test = train_test_split(
        X, y, test_size=0.15, stratify=y, random_state=SEED
    )
    # del 85% restante, 0.15/0.85 = 17.65% -> val = 15% del total
    X_train, X_val, y_train, y_val = train_test_split(
        X_tmp, y_tmp, test_size=0.15 / 0.85, stratify=y_tmp, random_state=SEED
    )
    del X, y, X_tmp, y_tmp
    gc.collect()

    # --- Escalado SIN fuga de informacion ---
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)   # fit + transform: SOLO Train
    X_val = scaler.transform(X_val).astype(np.float32)           # transform: usa params de Train
    X_test = scaler.transform(X_test).astype(np.float32)         # transform: usa params de Train

    np.savez(
        SPLITS_PATH,
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        X_test=X_test, y_test=y_test,
    )
    joblib.dump(scaler, SCALER_PATH)
    print(f"Splits guardados en {SPLITS_PATH} ({SPLITS_PATH.stat().st_size/1e6:.0f} MB)")

total = len(X_train) + len(X_val) + len(X_test)
print(f"\n{'Particion':<12}{'Filas':>12}{'%':>8}{'% clase 1 (senal)':>22}")
print("-" * 54)
for nombre, Xs, ys in [("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    print(f"{nombre:<12}{len(Xs):>12,}{100*len(Xs)/total:>7.1f}%{100*ys.mean():>21.3f}%")

print(f"\nVerificacion de escalado (Train):      media={X_train.mean():+.6f}  std={X_train.std():.6f}")
print(f"Verificacion de escalado (Validation): media={X_val.mean():+.6f}  std={X_val.std():.6f}")
print(f"Verificacion de escalado (Test):       media={X_test.mean():+.6f}  std={X_test.std():.6f}")
print("\nLa media ~0 y std ~1 en Train es exacta por construccion. Que Val y Test tengan valores")
print("MUY cercanos pero NO identicos es la evidencia de que fueron transformados con los")
print("parametros del Train y no re-ajustados: eso confirma la ausencia de Data Leakage.")

mem_report("tras crear splits")

---
# Celda 7 — Auditoría comparativa de particiones con `ydata-profiling`

Entregable obligatorio del enunciado. Se genera un reporte por partición y se comparan con
`compare([...])` para verificar tres cosas:

- **Ausencia de *data drift*:** las distribuciones de probabilidad de las 28 variables deben ser
  equivalentes entre Train, Validation y Test.
- **Estratificación respetada:** misma proporción de clases en las tres.
- **Escalado correcto:** media ≈ 0 y varianza ≈ 1, sin fugas.

**Cuidado crítico de memoria:** con 1M de filas, `ProfileReport` consumiría toda la RAM. Por eso
se usa `minimal=True` (desactiva correlaciones costosas e interacciones) y una submuestra de
**10.000 filas por partición**, tal como exige el enunciado.

In [ ]:
from ydata_profiling import ProfileReport, compare

N_PERFIL = 10_000
rng = np.random.default_rng(SEED)


def submuestra_df(X, y, n=N_PERFIL):
    '''DataFrame pequeno y representativo de una particion ya escalada.'''
    idx = rng.choice(len(X), size=min(n, len(X)), replace=False)
    d = pd.DataFrame(X[idx], columns=FEATURES)
    d["label"] = y[idx]
    return d


t0 = time.time()
rep_train = ProfileReport(submuestra_df(X_train, y_train), title="Train", minimal=True, progress_bar=False)
rep_val = ProfileReport(submuestra_df(X_val, y_val), title="Validation", minimal=True, progress_bar=False)
rep_test = ProfileReport(submuestra_df(X_test, y_test), title="Test", minimal=True, progress_bar=False)

comparacion = compare([rep_train, rep_val, rep_test])
HTML_PATH = REPORT_DIR / "comparacion_particiones.html"
comparacion.to_file(HTML_PATH)

print(f"Reporte comparativo generado en {time.time()-t0:.0f}s")
print(f"Archivo: {HTML_PATH}  ({HTML_PATH.stat().st_size/1e6:.1f} MB)")
print("--> Este .html es uno de los entregables que se adjuntan al correo.")

# Liberacion inmediata: los objetos ProfileReport son pesados
del rep_train, rep_val, rep_test, comparacion
gc.collect()
mem_report("tras ydata-profiling")

### Análisis escrito de la auditoría

> **(Completar tras ejecutar la celda anterior, revisando el `.html`.)** Debe afirmarse
> explícitamente: (a) que las distribuciones de las tres particiones son equivalentes — sin *data
> drift*; (b) que la proporción de clases se mantiene por la estratificación; y (c) que el
> escalado da media ≈ 0 y varianza ≈ 1 sin fugas de información.

---
# Celda 8 — EDA sobre submuestra del 10 %

A partir de aquí se trabaja con **100.000 filas (10 % de la submuestra)**, como exige el enunciado:
calcular histogramas, correlaciones, PCA o t-SNE sobre el millón completo agotaría la RAM sin
aportar información adicional — una submuestra aleatoria de 100k estima las distribuciones con
precisión más que suficiente.

Se usan los datos **sin escalar** (`df`), porque las unidades físicas originales son las
interpretables.

In [ ]:
FRAC_EDA = 0.10
df_eda = df.sample(frac=FRAC_EDA, random_state=SEED).reset_index(drop=True)
print(f"Submuestra de EDA: {len(df_eda):,} filas ({FRAC_EDA:.0%} del millon)")

# --- Balance de clases ---
conteo = df_eda["label"].value_counts().sort_index()
prop = df_eda["label"].value_counts(normalize=True).sort_index()
etiquetas = ["0 — Fondo (background)", "1 — Senal (Higgs)"]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(etiquetas, conteo.values, color=["#94a3b8", "#3b82f6"], edgecolor="black")
ax[0].set_title("Balance de clases (submuestra 10%)")
ax[0].set_ylabel("N° de eventos")
for i, v in enumerate(conteo.values):
    ax[0].text(i, v, f"{v:,}\n({prop.values[i]:.1%})", ha="center", va="bottom", fontsize=9)
ax[0].set_ylim(0, conteo.max() * 1.18)

ax[1].pie(conteo.values, labels=etiquetas, autopct="%1.2f%%", colors=["#94a3b8", "#3b82f6"],
          wedgeprops={"edgecolor": "black"})
ax[1].set_title("Proporcion")
plt.tight_layout()
plt.savefig(REPORT_DIR / "balance_clases.png", dpi=120, bbox_inches="tight")
plt.show()

ratio = conteo.max() / conteo.min()
print(f"\nRazon de desbalance: {ratio:.3f} : 1")
print("HIGGS esta practicamente balanceado, a diferencia de la Opcion A (fraude). Por lo tanto:")
print("  - Accuracy SI es una metrica informativa aqui, pero se reportara junto a ROC-AUC,")
print("    PR-AUC, F1 y la matriz de confusion para un analisis completo.")
print(f"  - pos_weight en BCEWithLogitsLoss = n_neg/n_pos = {conteo[0]/conteo[1]:.4f} (~1, correccion marginal).")

In [ ]:
# --- Histogramas de las 28 variables ---
fig, axes = plt.subplots(6, 5, figsize=(20, 20))
axes = axes.ravel()

for i, col in enumerate(FEATURES):
    ax = axes[i]
    ax.hist(df_eda.loc[df_eda.label == 0, col], bins=50, alpha=0.6, label="Fondo",
            color="#94a3b8", density=True)
    ax.hist(df_eda.loc[df_eda.label == 1, col], bins=50, alpha=0.6, label="Senal",
            color="#3b82f6", density=True)
    ax.set_title(col, fontsize=10)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

for j in range(len(FEATURES), len(axes)):
    axes[j].axis("off")

fig.suptitle("Distribucion de cada variable por clase (submuestra 10%)", fontsize=15, y=0.995)
plt.tight_layout()
plt.savefig(REPORT_DIR / "histogramas_variables.png", dpi=100, bbox_inches="tight")
plt.show()

print("Lectura: las variables de BAJO nivel (pt, eta, phi) se superponen casi por completo entre")
print("clases — individualmente casi no discriminan. Las de ALTO nivel (m_bb, m_wwbb, m_wbb)")
print("muestran separaciones visibles. Esto justifica el uso de una red profunda: la senal esta")
print("en las INTERACCIONES no lineales entre variables, no en ninguna variable aislada.")

In [ ]:
# --- Matriz de correlacion de Pearson ---
corr = df_eda[FEATURES].corr(method="pearson")

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES)))
ax.set_yticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=90, fontsize=8)
ax.set_yticklabels(FEATURES, fontsize=8)
ax.set_title("Matriz de correlacion de Pearson (submuestra 10%)", fontsize=13, pad=12)
fig.colorbar(im, ax=ax, shrink=0.8, label="Coeficiente de correlacion")
plt.tight_layout()
plt.savefig(REPORT_DIR / "matriz_correlacion.png", dpi=120, bbox_inches="tight")
plt.show()

# Pares mas colineales
cm = corr.abs().copy()
np.fill_diagonal(cm.values, 0.0)
pares = (
    cm.unstack().sort_values(ascending=False)
    .drop_duplicates().head(10)
    .rename("corr_abs").reset_index()
    .rename(columns={"level_0": "variable_1", "level_1": "variable_2"})
)
print("Pares de variables mas correlacionados:")
display(pares)
print("\nLa alta colinealidad entre las masas invariantes es esperable (se calculan a partir de las")
print("mismas magnitudes de bajo nivel) y es exactamente el motivo por el que el enunciado exige")
print("regularizacion explicita: Dropout elevado, BatchNorm y Weight Decay en AdamW.")

del corr, cm
gc.collect()

---
# Celda 9 — Análisis No Supervisado: PCA, t-SNE y K-Means

Antes de entrenar cualquier red, conviene preguntarse: **¿las dos clases forman grupos separables
en el espacio de las variables?** Si un método no supervisado (que no ve las etiquetas) recuperara
la separación señal/fondo, el problema sería fácil. El resultado esperado aquí es el contrario, y
eso es informativo: justifica la necesidad de un modelo profundo.

Se trabaja sobre los datos **escalados** (PCA, t-SNE y K-Means son sensibles a la escala) y sobre
submuestras aún menores, porque t-SNE tiene costo cuadrático.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Submuestra escalada para el analisis no supervisado
N_NS = 20_000
idx_ns = rng.choice(len(X_train), size=N_NS, replace=False)
X_ns, y_ns = X_train[idx_ns], y_train[idx_ns]
print(f"Submuestra no supervisada: {X_ns.shape}")

# ---------- PCA ----------
pca_full = PCA(n_components=len(FEATURES), random_state=SEED).fit(X_ns)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = int(np.searchsorted(var_acum, 0.90) + 1)

X_pca = PCA(n_components=2, random_state=SEED).fit_transform(X_ns)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(range(1, len(var_acum) + 1), var_acum, "o-", color="#3b82f6", markersize=4)
ax[0].axhline(0.90, color="red", ls="--", lw=1, label="90% de varianza")
ax[0].axvline(n_90, color="green", ls="--", lw=1, label=f"{n_90} componentes")
ax[0].set_xlabel("N° de componentes principales")
ax[0].set_ylabel("Varianza explicada acumulada")
ax[0].set_title("PCA — varianza explicada")
ax[0].legend()
ax[0].grid(alpha=0.3)

for cls, color, lbl in [(0, "#94a3b8", "Fondo"), (1, "#3b82f6", "Senal")]:
    m = y_ns == cls
    ax[1].scatter(X_pca[m, 0], X_pca[m, 1], s=3, alpha=0.3, c=color, label=lbl)
ax[1].set_xlabel("PC 1"); ax[1].set_ylabel("PC 2")
ax[1].set_title("Proyeccion PCA (2 componentes)")
ax[1].legend(markerscale=4)
plt.tight_layout()
plt.savefig(REPORT_DIR / "pca.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Se requieren {n_90} de {len(FEATURES)} componentes para explicar el 90% de la varianza.")
print(f"Las 2 primeras solo explican {100*var_acum[1]:.1f}%: la informacion esta repartida, no hay")
print("un subespacio lineal dominante y una reduccion lineal agresiva destruiria senal util.")

del pca_full
gc.collect()

In [ ]:
# ---------- t-SNE ----------
# Costo cuadratico: se limita a 5.000 puntos.
N_TSNE = 5_000
idx_t = rng.choice(N_NS, size=N_TSNE, replace=False)

t0 = time.time()
X_tsne = TSNE(
    n_components=2, perplexity=30, learning_rate="auto",
    init="pca", random_state=SEED,
).fit_transform(X_ns[idx_t])
print(f"t-SNE calculado en {time.time()-t0:.0f}s sobre {N_TSNE:,} puntos")

fig, ax = plt.subplots(figsize=(7, 6))
for cls, color, lbl in [(0, "#94a3b8", "Fondo"), (1, "#3b82f6", "Senal")]:
    m = y_ns[idx_t] == cls
    ax.scatter(X_tsne[m, 0], X_tsne[m, 1], s=5, alpha=0.5, c=color, label=lbl)
ax.set_title("t-SNE (perplexity=30)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(REPORT_DIR / "tsne.png", dpi=120, bbox_inches="tight")
plt.show()

print("No aparecen dos islas limpias: senal y fondo estan entremezclados incluso bajo una")
print("proyeccion no lineal. La frontera de decision es compleja y de alta dimension.")

In [ ]:
# ---------- K-Means ----------
N_SIL = 10_000  # silhouette_score es O(n^2) en memoria
idx_s = rng.choice(N_NS, size=N_SIL, replace=False)

resultados = []
for k in [2, 3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X_ns)
    sil = silhouette_score(X_ns[idx_s], km.labels_[idx_s])
    ari = adjusted_rand_score(y_ns, km.labels_) if k == 2 else np.nan
    resultados.append({"k": k, "silueta": sil, "ARI_vs_etiqueta_real": ari})
    del km
    gc.collect()

tabla = pd.DataFrame(resultados)
display(tabla.round(4))

km2 = KMeans(n_clusters=2, random_state=SEED, n_init=10).fit(X_ns)
ari2 = adjusted_rand_score(y_ns, km2.labels_)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(X_pca[:, 0], X_pca[:, 1], s=3, alpha=0.3, c=km2.labels_, cmap="coolwarm")
ax[0].set_title(f"K-Means (k=2) sobre proyeccion PCA — ARI = {ari2:.4f}")
ax[0].set_xlabel("PC 1"); ax[0].set_ylabel("PC 2")

for cls, color, lbl in [(0, "#94a3b8", "Fondo"), (1, "#3b82f6", "Senal")]:
    m = y_ns == cls
    ax[1].scatter(X_pca[m, 0], X_pca[m, 1], s=3, alpha=0.3, c=color, label=lbl)
ax[1].set_title("Etiquetas reales sobre la misma proyeccion")
ax[1].set_xlabel("PC 1"); ax[1].set_ylabel("PC 2")
ax[1].legend(markerscale=4)
plt.tight_layout()
plt.savefig(REPORT_DIR / "kmeans.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"ARI de K-Means (k=2) frente a la etiqueta real: {ari2:.4f}")
print("Un ARI cercano a 0 significa que la particion no supervisada NO coincide con la division")
print("senal/fondo: la estructura geometrica dominante de los datos (dictada por la cinematica de")
print("las colisiones) es independiente de la clase fisica que queremos predecir.")
print("\nCONCLUSION DE LA FASE A: el problema NO es separable por metodos lineales ni por")
print("agrupamiento por distancia. Se necesita aprendizaje supervisado profundo, que es")
print("exactamente la hipotesis del paper original de Baldi et al. (2014).")

del km2, X_ns, y_ns, X_pca, X_tsne
gc.collect()
mem_report("fin del analisis no supervisado")

---
# Celda 10 — Resumen de artefactos generados

Verificación final de que todo lo que necesitan los notebooks 02 y 03 quedó persistido.

In [ ]:
print(f"ARTEFACTOS GENERADOS EN {PROJECT_DIR}")
print("=" * 72)
for p in sorted(PROJECT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(PROJECT_DIR)):<45} {p.stat().st_size/1e6:>9.1f} MB")

print("\nLISTO PARA EL NOTEBOOK 02 (modelos base):")
print(f"  - Features: {len(FEATURES)}")
print(f"  - Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"  - Clases: binario (0 = fondo, 1 = senal)")
mem_report("final")

---
# Conclusiones de la Fase A

> **(Completar tras la ejecución.)** Puntos a cubrir:
>
> 1. **Gestión de memoria:** cuánta RAM llegó a usarse en el pico frente a los ~12 GB de Colab, y
>    qué habría pasado leyendo el CSV completo de 7.5 GB de una sola vez.
> 2. **Auditoría de particiones:** confirmación de homogeneidad entre Train/Val/Test y ausencia de
>    *data leakage*, citando el `.html` de ydata-profiling.
> 3. **Hallazgos del EDA:** balance de clases, variables de alto nivel como las más discriminantes,
>    colinealidad y su implicancia en la regularización.
> 4. **Análisis no supervisado:** por qué un ARI ≈ 0 y una proyección t-SNE entremezclada
>    justifican la arquitectura profunda y el ensamble que se construyen en los notebooks 02 y 03.